# PathSnap — Model 3: K-Nearest Neighbours
**Owner:** Pandu

Instance-based: a candidate is scored by the labels of the 25 most similar training candidates (distance-weighted). Needs standardised features; fitted on a 300k-row subsample because prediction cost grows with stored rows.

Evaluation protocol (identical for every model):
* 5-fold **GroupKFold by trip** on the 80 training trips → CV Macro F1 (no trip in both train and validation fold)
* refit on all training data → evaluate on the 10 **student_test** trips: Macro F1, precision, recall, log loss,
  **observation-level Top-1 accuracy** (best-scoring candidate per GPS point == reference arc), inference time.

## 0. Setup (Kaggle)
1. **Add data**: upload the provided `Student folder` (with `train/` and `student_test/`) as a Kaggle Dataset and attach it with *Add Input*.
2. **Settings → Internet: On** (needed for `git clone`).

In [ ]:
!git clone -q https://github.com/GTX-Pradeep/Path_Snap_PT_096.git /kaggle/working/pathsnap || (cd /kaggle/working/pathsnap && git pull -q)
!pip install -q lightgbm optuna folium pyarrow
import os, sys, glob
# find the attached dataset folder that contains train/ and student_test/
data_dir = next(os.path.dirname(p) for p in glob.glob('/kaggle/input/**/train', recursive=True)
                if os.path.isdir(os.path.join(os.path.dirname(p), 'student_test')))
os.environ['PATHSNAP_DATA'] = data_dir
os.chdir('/kaggle/working/pathsnap/src'); sys.path.insert(0, '.')
print('data:', data_dir, os.listdir(data_dir))

## 1. Build candidate tables (8 nearest arcs per GPS point + features + reference labels)

In [ ]:
import os
if not os.path.exists('../results/candidates_student_test.parquet'):
    !python build_dataset.py
import pandas as pd
from train_models import load_tables
from features import FEATURES
tr, te = load_tables()
print('train rows (after removing obs with unreliable reference >50 m):', len(tr), '| student_test rows:', len(te))
print('features:', FEATURES)
tr.head()

## 2. Train + evaluate `knn`

In [ ]:
import json
from train_models import evaluate, MODELS
print(MODELS['knn']['make']())
res = evaluate('knn', tr=tr, te=te)
{k: v for k, v in res.items() if k not in ('config', 'loss_curve', 'validation_scores')}

## 3. Diagnostics: confusion matrix, CV vs train vs test (over/under-fitting check)

In [ ]:
!python plot_results.py knn
from IPython.display import Image
Image('../results/plots/knn_diagnostics.png')

## 5. Comparison with all models trained so far

In [ ]:
from train_models import comparison_table
!python plot_results.py
display(comparison_table())
Image('../results/plots/model_comparison.png')